In [1]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [2]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("FRIKSTAD_processed.csv")
df_weather = load_blob_csv("FRIKSTAD_weather.csv")
df_norgespris = load_blob_csv("frikstad_norgespris.csv")

In [3]:
# Del df_consumption i to deler uten å endre resten av analyseflyten.
# Dette beholder samme variabelnavn for analysen lenger ned.
split_index = len(df_consumption) // 2

df_consumption_part1 = df_consumption.iloc[:split_index].copy()
df_consumption_part2 = df_consumption.iloc[split_index:].copy()

df_consumption = pd.concat([df_consumption_part1, df_consumption_part2], ignore_index=True)

print(f"df_consumption delt i to deler: {len(df_consumption_part1):,} + {len(df_consumption_part2):,} rader")

df_consumption delt i to deler: 13,643,232 + 13,643,232 rader


In [4]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.regression import (
    prepare_norgespris_regression_data,
    fit_best_norgespris_model
)

regression_df = prepare_norgespris_regression_data(
    df_consumption_part1,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)

results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print("Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.")
print(f"Gjennomsnittlig andel med Norgespris i etterperioden: {metrics['mean_share_post']:.2%}")
print(f"Effekt ved +10 prosentpoeng høyere andel: {metrics['effect_10pp_pct']:.2f}%")
print(f"Effekt ved 0% -> 100% andel: {metrics['effect_full_share_pct']:.2f}%")
print()
print(f"Total observert forbruk i etterperioden: {metrics['total_observed_post_kwh']:,.0f} kWh")
print(f"Estimert total effekt av Norgespris i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Dette tilsvarer: {metrics['attributable_share_of_post_kwh_pct']:.2f}% av totalforbruket i etterperioden")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag (etterperiode): {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")
print()
print(results.summary)
print()
print(f"Koeffisient for norgespris_share (log-skala): {metrics['beta_share']:.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.4g}")

Observasjoner totalt: 12,277,841
Brukt i modell: 12,277,841
R2 (within): 0.1347
Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.
Gjennomsnittlig andel med Norgespris i etterperioden: 78.83%
Effekt ved +10 prosentpoeng høyere andel: 0.20%
Effekt ved 0% -> 100% andel: 1.98%

Total observert forbruk i etterperioden: 12,905,517 kWh
Estimert total effekt av Norgespris i etterperioden: 285,080 kWh
Dette tilsvarer: 2.21% av totalforbruket i etterperioden
Per kunde i etterperioden: 46.4 kWh
Per kunde per dag (etterperiode): 0.594 kWh

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1347
Estimator:                   PanelOLS   R-squared (Between):              0.0012
No. Observations:            12277841   R-squared (Within):               0.1347
Date:                Thu, Apr 23 2026   R-squared (Overall):              0.0469
Time:            

In [5]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../src")))

from analysis.regression import (
    prepare_norgespris_regression_data,
    fit_best_norgespris_model
)

regression_df = prepare_norgespris_regression_data(
    df_consumption_part2,
    df_weather,
    df_norgespris,
    station_customers_total=None,
    exclude_consumption_codes=(26,),
)

results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print("Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.")
print(f"Gjennomsnittlig andel med Norgespris i etterperioden: {metrics['mean_share_post']:.2%}")
print(f"Effekt ved +10 prosentpoeng høyere andel: {metrics['effect_10pp_pct']:.2f}%")
print(f"Effekt ved 0% -> 100% andel: {metrics['effect_full_share_pct']:.2f}%")
print()
print(f"Total observert forbruk i etterperioden: {metrics['total_observed_post_kwh']:,.0f} kWh")
print(f"Estimert total effekt av Norgespris i etterperioden: {metrics['total_attributable_post_kwh']:,.0f} kWh")
print(f"Dette tilsvarer: {metrics['attributable_share_of_post_kwh_pct']:.2f}% av totalforbruket i etterperioden")
print(f"Per kunde i etterperioden: {metrics['attributable_per_customer_post_kwh']:.1f} kWh")
print(f"Per kunde per dag (etterperiode): {metrics['attributable_per_customer_post_day_kwh']:.3f} kWh")
print()
print(results.summary)
print()
print(f"Koeffisient for norgespris_share (log-skala): {metrics['beta_share']:.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris_share'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris_share'):.4g}")

Observasjoner totalt: 12,330,457
Brukt i modell: 12,330,457
R2 (within): 0.1374
Merk: modellen bruker log1p(value_kwh) = log(1 + value_kwh), så effekten gjelder 1 + forbruk.
Gjennomsnittlig andel med Norgespris i etterperioden: 78.23%
Effekt ved +10 prosentpoeng høyere andel: 0.19%
Effekt ved 0% -> 100% andel: 1.92%

Total observert forbruk i etterperioden: 12,906,438 kWh
Estimert total effekt av Norgespris i etterperioden: 277,003 kWh
Dette tilsvarer: 2.15% av totalforbruket i etterperioden
Per kunde i etterperioden: 45.0 kWh
Per kunde per dag (etterperiode): 0.578 kWh

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1374
Estimator:                   PanelOLS   R-squared (Between):             -0.0008
No. Observations:            12330457   R-squared (Within):               0.1374
Date:                Thu, Apr 23 2026   R-squared (Overall):              0.0474
Time:            

In [6]:
import numpy as np

np.set_printoptions(suppress=False)
print(results.pvalues)

Intercept           0.000000e+00
norgespris_share    2.220446e-16
air_temperature     0.000000e+00
wind_speed          0.000000e+00
precipitation_mm    0.000000e+00
is_weekend          0.000000e+00
is_holiday          0.000000e+00
C(hour)[T.1]        0.000000e+00
C(hour)[T.2]        0.000000e+00
C(hour)[T.3]        0.000000e+00
C(hour)[T.4]        0.000000e+00
C(hour)[T.5]        2.137419e-08
C(hour)[T.6]        0.000000e+00
C(hour)[T.7]        0.000000e+00
C(hour)[T.8]        0.000000e+00
C(hour)[T.9]        0.000000e+00
C(hour)[T.10]       0.000000e+00
C(hour)[T.11]       0.000000e+00
C(hour)[T.12]       0.000000e+00
C(hour)[T.13]       0.000000e+00
C(hour)[T.14]       0.000000e+00
C(hour)[T.15]       0.000000e+00
C(hour)[T.16]       0.000000e+00
C(hour)[T.17]       0.000000e+00
C(hour)[T.18]       0.000000e+00
C(hour)[T.19]       0.000000e+00
C(hour)[T.20]       0.000000e+00
C(hour)[T.21]       0.000000e+00
C(hour)[T.22]       0.000000e+00
C(hour)[T.23]       0.000000e+00
C(month)[T